In [1]:
!pip install mysql-connector-python pandas


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Anindita#2026",
    database="ecommerce"
)

cursor = conn.cursor()
print("Connected successfully")


Connected successfully


In [3]:
import pandas as pd

df = pd.read_csv("olist_orders_dataset.csv")
print(df.head())

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08

In [4]:
df = df.where(pd.notnull(df), None)   # convert NaN → NULL

# convert datetime safely
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'], errors='coerce')

In [5]:
query = """
INSERT INTO orders (
order_id, customer_id, order_status,
order_purchase_timestamp, order_approved_at,
order_delivered_carrier_date, order_delivered_customer_date,
order_estimated_delivery_date
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
"""

for _, row in df.iterrows():
    cursor.execute(query, tuple(row[:8]))

conn.commit()

In [8]:
import pandas as pd
import mysql.connector

def load_orders(file_path):
    try:
        # 🔹 Connect
        conn = mysql.connector.connect(
            host="localhost",
            user="root",
            password="Anindita#2026",
            database="ecommerce"
        )
        cursor = conn.cursor()

        print("Connected to DB")

        # 🔹 Extract
        df = pd.read_csv(file_path)

        # 🔹 Select only required columns (SAFE)
        df = df[[
            'order_id',
            'customer_id',
            'order_status',
            'order_purchase_timestamp',
            'order_approved_at',
            'order_delivered_carrier_date',
            'order_delivered_customer_date',
            'order_estimated_delivery_date'
        ]]

        # 🔹 Transform
        df = df.where(pd.notnull(df), None)

        # Convert datetime safely
        for col in df.columns[3:]:
            df[col] = pd.to_datetime(df[col], errors='coerce')

        # 🔹 Convert to tuple
        data = [tuple(row) for row in df.to_numpy()]

        # 🔹 Query with column names (BEST PRACTICE)
        query = """
        INSERT INTO orders (
            order_id,
            customer_id,
            order_status,
            order_purchase_timestamp,
            order_approved_at,
            order_delivered_carrier_date,
            order_delivered_customer_date,
            order_estimated_delivery_date
        )
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
        """

        # 🔹 Load
        cursor.executemany(query, data)
        conn.commit()

        print("✅ Data Loaded Successfully:", len(data), "rows")

    except Exception as e:
        print("❌ Error:", e)

    finally:
        conn.close()
        print("Connection closed")

In [9]:
load_orders("D:/Data/E-commerce Data Pipeline + Analytics System/olist_orders_dataset.csv")

Connected to DB
✅ Data Loaded Successfully: 99441 rows
Connection closed


In [10]:
df['year'] = df['order_purchase_timestamp'].dt.year
df['month'] = df['order_purchase_timestamp'].dt.month